In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline

In [2]:
from sklearn.feature_extraction.text import (
    CountVectorizer,
    TfidfTransformer,
)

In [3]:
from datasets import load_dataset

ds = load_dataset("seeeeiii/RICO-WidgetCaptioning")

c:\Users\Mark\ml\screenshot-classifier\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
ds

DatasetDict({
    train: Dataset({
        features: ['screenId', 'captions', 'view_hierarchy', 'bbox', 'file_name', 'file_name_semantic', 'semantic_annotations', 'app_package_name', 'play_store_name', 'category', 'average_rating', 'number_of_ratings', 'number_of_downloads', 'file_name_icon', 'image', 'image_icon', 'image_semantic'],
        num_rows: 41221
    })
    val: Dataset({
        features: ['screenId', 'captions', 'view_hierarchy', 'bbox', 'file_name', 'file_name_semantic', 'semantic_annotations', 'app_package_name', 'play_store_name', 'category', 'average_rating', 'number_of_ratings', 'number_of_downloads', 'file_name_icon', 'image', 'image_icon', 'image_semantic'],
        num_rows: 3483
    })
    test: Dataset({
        features: ['screenId', 'captions', 'view_hierarchy', 'bbox', 'file_name', 'file_name_semantic', 'semantic_annotations', 'app_package_name', 'play_store_name', 'category', 'average_rating', 'number_of_ratings', 'number_of_downloads', 'file_name_icon', 'ima

In [5]:
train_ds = ds['train']
valid_ds = ds['val']
test_ds = ds['test']

In [6]:
excluded_categories = {"Events", "000 - 1"}

def remove_categories(dataset):
    return dataset.filter(
        lambda category: category not in excluded_categories,
        input_columns=["category"],
    )

train_ds = remove_categories(train_ds)
valid_ds = remove_categories(valid_ds)
test_ds = remove_categories(test_ds)

In [7]:
del ds

In [8]:
def simplify_ds(ds):
  compact = ds.select_columns(
      ["screenId", "captions", "category", "app_package_name"]
  ).rename_columns({
      "screenId": "id",
      "app_package_name": "package_name",
  })
  return compact

In [9]:
categories = train_ds.unique("category")

print("Number of categories:", len(categories))
print(categories)

Number of categories: 26
['Travel & Local', 'Shopping', 'Communication', 'Lifestyle', 'Maps & Navigation', 'Education', 'Sports', 'Social', 'Comics', 'Music & Audio', 'House & Home', 'Books & Reference', 'News & Magazines', 'Auto & Vehicles', 'Finance', 'Health & Fitness', 'Weather', 'Parenting', 'Video Players & Editors', 'Business', 'Dating', 'Medical', 'Food & Drink', 'Entertainment', 'Art & Design', 'Beauty']


In [10]:
train_ds = simplify_ds(train_ds)
valid_ds = simplify_ds(valid_ds)
test_ds = simplify_ds(test_ds)

In [11]:
def split_ds(ds):
  return ds.map(
    lambda row: {
      "text": "|".join(row['captions']),
      "label": row['category']
    }
  )

In [12]:
train_ds = split_ds(train_ds)
valid_ds = split_ds(valid_ds)
test_ds = split_ds(test_ds)
train_ds

Dataset({
    features: ['id', 'captions', 'category', 'package_name', 'text', 'label'],
    num_rows: 41148
})

In [13]:
X_train, y_train = train_ds["text"], train_ds["label"]
X_valid, y_valid = valid_ds["text"], valid_ds["label"]
X_test, y_test = test_ds["text"], test_ds["label"]


In [14]:
len(X_train), len(y_train)

(41148, 41148)

In [15]:
from collections import Counter

train_counts = Counter(y_train)

for category, count in train_counts.most_common():
    print(f"{category}: {count}")

Entertainment: 3699
Health & Fitness: 2957
Education: 2849
Lifestyle: 2764
Social: 2586
Music & Audio: 2419
Shopping: 2331
Communication: 2245
Books & Reference: 2195
News & Magazines: 1899
Travel & Local: 1809
Sports: 1763
Finance: 1447
Weather: 1402
Medical: 1152
Business: 1111
Video Players & Editors: 996
Maps & Navigation: 950
Food & Drink: 917
Comics: 794
Parenting: 686
Dating: 681
Auto & Vehicles: 487
Beauty: 356
Art & Design: 335
House & Home: 318


In [16]:

most_common_category, count = train_counts.most_common(1)[0]
baseline_accuracy = count / len(X_train)
baseline_accuracy

0.08989501312335958

In [17]:
baseline_predictions = [
    most_common_category
    for _ in y_valid
]

validation_baseline = accuracy_score(
    y_valid,
    baseline_predictions,
)

validation_baseline

0.044270833333333336

In [18]:
X_train[0], y_train[0]

('get more information|go to options|more information', 'Travel & Local')

In [36]:
count_vectorizer = CountVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
)

count_matrix = count_vectorizer.fit_transform(X_train)

print(count_vectorizer.get_feature_names_out())
print(count_matrix.toarray())

tfidf_transformer = TfidfTransformer(
    norm="l2",
    use_idf=True,
    smooth_idf=True,
    sublinear_tf=False,
)

X_train_tfidf = tfidf_transformer.fit_transform(count_matrix)

X_train_tfidf.shape

['00' '000' '03' ... 'zoon' 'zoon in' 'zoosk']
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


(41148, 23955)

In [34]:
X_train_tfidf.shape

(41148, 23955)

In [20]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_valid_tfidf = vectorizer.transform(X_valid)

In [21]:
print(X_train_tfidf.shape)
print(vectorizer.get_feature_names_out()[20:])

(41148, 23955)
['13' '14' '14th' ... 'zoon' 'zoon in' 'zoosk']


In [22]:
model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42,
)

In [23]:
model.fit(X_train_tfidf, y_train)

valid_predictions = model.predict(X_valid_tfidf)

print("Validation accuracy:", accuracy_score(y_valid, valid_predictions))
print(classification_report(y_valid, valid_predictions))

Validation accuracy: 0.21238425925925927
                         precision    recall  f1-score   support

           Art & Design       0.40      0.69      0.50       123
        Auto & Vehicles       0.01      0.06      0.02        16
                 Beauty       0.03      0.15      0.04        13
      Books & Reference       0.15      0.12      0.13       164
               Business       0.09      0.23      0.13        80
                 Comics       0.06      0.12      0.08        59
          Communication       0.40      0.24      0.30       264
                 Dating       0.09      0.20      0.13        55
              Education       0.14      0.15      0.14       108
          Entertainment       0.31      0.13      0.18       153
                Finance       0.20      0.25      0.22       108
           Food & Drink       0.21      0.22      0.22       153
       Health & Fitness       0.24      0.13      0.17       172
           House & Home       0.02      0.38    

In [24]:
for text, actual, predicted in zip(X_valid[:10], y_valid[:10], valid_predictions[:10]):
    print(f"Text:      {text}")
    print(f"Actual:    {actual}")
    print(f"Predicted: {predicted}")
    print()

Text:      look for|search|search
Actual:    Maps & Navigation
Predicted: Shopping

Text:      contacts|go to friends
Actual:    Maps & Navigation
Predicted: Communication

Text:      browse worldwide|world search
Actual:    Maps & Navigation
Predicted: Maps & Navigation

Text:      go to settings|settings
Actual:    Maps & Navigation
Predicted: Communication

Text:      enter e-mail address|name|type email address
Actual:    Shopping
Predicted: Food & Drink

Text:      enter password|enter password|insert password
Actual:    Shopping
Predicted: Finance

Text:      get suggestions about this feature|help|info on saving email
Actual:    Shopping
Predicted: Health & Fitness

Text:      add email to autofill|save email
Actual:    Shopping
Predicted: Parenting

Text:      search by phone number or name|send message
Actual:    Communication
Predicted: Communication

Text:      favorite chicken stuffed kulcha|favorite recipe|like button
Actual:    Food & Drink
Predicted: Food & Drink



In [25]:
most_common_category, count = train_counts.most_common(1)[0]
baseline_accuracy = count / len(y_train)

print(most_common_category)
print(f"Baseline accuracy: {baseline_accuracy:.2%}")

Entertainment
Baseline accuracy: 8.99%


In [26]:
print(set(y_valid) - set(y_train))

set()


In [27]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
    )),
])

In [28]:
pipeline.fit(X_train, y_train)

valid_predictions = pipeline.predict(X_valid)

print(f"Validation accuracy: {accuracy_score(y_valid, valid_predictions):.2%}")
print(classification_report(y_valid, valid_predictions))

Validation accuracy: 21.24%
                         precision    recall  f1-score   support

           Art & Design       0.40      0.69      0.50       123
        Auto & Vehicles       0.01      0.06      0.02        16
                 Beauty       0.03      0.15      0.04        13
      Books & Reference       0.15      0.12      0.13       164
               Business       0.09      0.23      0.13        80
                 Comics       0.06      0.12      0.08        59
          Communication       0.40      0.24      0.30       264
                 Dating       0.09      0.20      0.13        55
              Education       0.14      0.15      0.14       108
          Entertainment       0.31      0.13      0.18       153
                Finance       0.20      0.25      0.22       108
           Food & Drink       0.21      0.22      0.22       153
       Health & Fitness       0.24      0.13      0.17       172
           House & Home       0.02      0.38      0.04       

In [29]:
for text, actual, predicted in zip(X_valid[:10], y_valid[:10], valid_predictions[:10]):
    print(f"Caption:   {text}")
    print(f"Actual:    {actual}")
    print(f"Predicted: {predicted}\n")

Caption:   look for|search|search
Actual:    Maps & Navigation
Predicted: Shopping

Caption:   contacts|go to friends
Actual:    Maps & Navigation
Predicted: Communication

Caption:   browse worldwide|world search
Actual:    Maps & Navigation
Predicted: Maps & Navigation

Caption:   go to settings|settings
Actual:    Maps & Navigation
Predicted: Communication

Caption:   enter e-mail address|name|type email address
Actual:    Shopping
Predicted: Food & Drink

Caption:   enter password|enter password|insert password
Actual:    Shopping
Predicted: Finance

Caption:   get suggestions about this feature|help|info on saving email
Actual:    Shopping
Predicted: Health & Fitness

Caption:   add email to autofill|save email
Actual:    Shopping
Predicted: Parenting

Caption:   search by phone number or name|send message
Actual:    Communication
Predicted: Communication

Caption:   favorite chicken stuffed kulcha|favorite recipe|like button
Actual:    Food & Drink
Predicted: Food & Drink



In [30]:
from pathlib import Path
import json

from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import StringTensorType

export_dir = Path("models")
export_dir.mkdir(exist_ok=True)

tfidf_options = {
    "tfidf": {
        "separators": [
            " ", "[.]", "\\?", ",", ";", ":", "\\!",
            "\\(", "\\)", "\\[", "\\]", "\\|", "\\-", "\n",
            "\"", "'",
        ]
    },
    "classifier": {
        "zipmap": False,
        "nocl": True,
    },
}

onnx_model = convert_sklearn(
    pipeline,
    initial_types=[("text", StringTensorType([None, 1]))],
    options=tfidf_options,
    target_opset=17,
)

onnx_path = export_dir / "category_classifier.onnx"
onnx_path.write_bytes(onnx_model.SerializeToString())

labels_path = export_dir / "labels.json"
labels_path.write_text(
    json.dumps(pipeline.named_steps["classifier"].classes_.tolist()),
    encoding="utf-8",
)

print(onnx_path.resolve())
print(labels_path.resolve())

C:\Users\Mark\ml\screenshot-classifier\src\notebooks\models\category_classifier.onnx
C:\Users\Mark\ml\screenshot-classifier\src\notebooks\models\labels.json


In [31]:
import numpy as np
import onnxruntime as ort

sample_texts = X_valid[:10]

session = ort.InferenceSession(
    "models/category_classifier.onnx",
    providers=["CPUExecutionProvider"],
)

onnx_inputs = np.asarray(sample_texts, dtype=object).reshape(-1, 1)
outputs = session.run(None, {"text": onnx_inputs})

print([output.shape for output in outputs if hasattr(output, "shape")])
print("Python:", pipeline.predict(sample_texts))

[(10,), (10, 26)]
Python: ['Shopping' 'Communication' 'Maps & Navigation' 'Communication'
 'Food & Drink' 'Finance' 'Health & Fitness' 'Parenting' 'Communication'
 'Food & Drink']


In [32]:
print([item.name for item in session.get_inputs()])
print([item.name for item in session.get_outputs()])

['text']
['label', 'probabilities']
